# HealthIQ — 02: Exploratory Data Analysis

**Objective:** Explore patient demographics, hospital operations, financials, outcomes, and variable relationships.

**Input:** `../data/processed/cleaned_healthcare_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('../data/processed/cleaned_healthcare_data.csv', parse_dates=['visit_date'])
print(f'Shape: {df.shape}')
df.head()

---
## 1. Patient Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Age group
age_order = ['18-30', '31-45', '46-60', '60+']
age_counts = df['age_group'].value_counts().reindex(age_order)
axes[0].bar(age_counts.index, age_counts.values, color=sns.color_palette('muted', 4))
axes[0].set_title('Patients by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')
for i, v in enumerate(age_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=10)

# Gender
gender_counts = df['gender'].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values, color=['steelblue', 'salmon'])
axes[1].set_title('Patients by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Count')
for i, v in enumerate(gender_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=10)

# Region
region_counts = df['region'].value_counts()
axes[2].bar(region_counts.index, region_counts.values, color=sns.color_palette('Set2', 4))
axes[2].set_title('Patients by Region')
axes[2].set_xlabel('Region')
axes[2].set_ylabel('Count')
for i, v in enumerate(region_counts.values):
    axes[2].text(i, v + 10, str(v), ha='center', fontsize=10)

plt.suptitle('Patient Demographics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/01_patient_demographics.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 2. Monthly Visit Trend

In [ ]:
monthly = df.groupby(df['visit_date'].dt.to_period('M')).size().reset_index()
monthly.columns = ['month', 'visits']
monthly['month_str'] = monthly['month'].astype(str)

plt.figure(figsize=(10, 5))
plt.plot(monthly['month_str'], monthly['visits'], marker='o', linewidth=2, color='steelblue')
plt.fill_between(monthly['month_str'], monthly['visits'], alpha=0.15, color='steelblue')
plt.title('Monthly Patient Visits (Jan–Jul 2022)', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Number of Visits')
plt.xticks(rotation=30)
plt.ylim(0, monthly['visits'].max() * 1.15)
for _, row in monthly.iterrows():
    plt.text(row['month_str'], row['visits'] + 10, str(row['visits']), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/charts/02_monthly_visits.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Department & Visit Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Department
dept_counts = df['department'].value_counts()
axes[0].barh(dept_counts.index, dept_counts.values, color=sns.color_palette('muted', len(dept_counts)))
axes[0].set_title('Visits by Department')
axes[0].set_xlabel('Number of Visits')
for i, v in enumerate(dept_counts.values):
    axes[0].text(v + 5, i, str(v), va='center', fontsize=10)

# Visit type
vt_counts = df['visit_type'].value_counts()
axes[1].pie(vt_counts.values, labels=vt_counts.index, autopct='%1.1f%%',
            colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Visit Type Distribution')

plt.suptitle('Hospital Operations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/03_department_visittype.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 4. Length of Stay

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['length_of_stay_days'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Length of Stay')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Count')
axes[0].axvline(df['length_of_stay_days'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["length_of_stay_days"].mean():.2f} days')
axes[0].legend()

# Boxplot by visit type
dept_order = df.groupby('visit_type')['length_of_stay_days'].mean().sort_values(ascending=False).index
df_plot = df[df['visit_type'].isin(dept_order)]
sns.boxplot(data=df, x='visit_type', y='length_of_stay_days', order=dept_order,
            palette='muted', ax=axes[1])
axes[1].set_title('LOS by Visit Type')
axes[1].set_xlabel('Visit Type')
axes[1].set_ylabel('Days')

plt.suptitle('Length of Stay Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/04_length_of_stay.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Treatment Cost Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of treatment cost
axes[0].hist(df['treatment_cost'], bins=40, color='teal', edgecolor='white')
axes[0].set_title('Distribution of Treatment Cost')
axes[0].set_xlabel('Treatment Cost (USD)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['treatment_cost'].mean(), color='red', linestyle='--',
                label=f'Mean: ${df["treatment_cost"].mean():,.0f}')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# Avg cost by department
avg_cost_dept = df.groupby('department')['treatment_cost'].mean().sort_values()
axes[1].barh(avg_cost_dept.index, avg_cost_dept.values, color=sns.color_palette('muted', len(avg_cost_dept)))
axes[1].set_title('Average Cost by Department')
axes[1].set_xlabel('Average Cost (USD)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
for i, v in enumerate(avg_cost_dept.values):
    axes[1].text(v + 100, i, f'${v:,.0f}', va='center', fontsize=9)

plt.suptitle('Treatment Cost Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/05_treatment_cost.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplot: cost by department
dept_order = df.groupby('department')['treatment_cost'].median().sort_values(ascending=False).index
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='department', y='treatment_cost', order=dept_order, palette='muted')
plt.title('Treatment Cost Distribution by Department', fontsize=13, fontweight='bold')
plt.xlabel('Department')
plt.ylabel('Treatment Cost (USD)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.tight_layout()
plt.savefig('../reports/charts/06_cost_boxplot_dept.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Recovery Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df['recovery_score'], bins=30, color='mediumseagreen', edgecolor='white')
axes[0].axvline(df['recovery_score'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["recovery_score"].mean():.2f}')
axes[0].set_title('Distribution of Recovery Score')
axes[0].set_xlabel('Recovery Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# Avg recovery by department
avg_rec = df.groupby('department')['recovery_score'].mean().sort_values()
axes[1].barh(avg_rec.index, avg_rec.values, color=sns.color_palette('Set2', len(avg_rec)))
axes[1].set_title('Avg Recovery Score by Department')
axes[1].set_xlabel('Average Recovery Score')
axes[1].set_xlim(70, 76)
for i, v in enumerate(avg_rec.values):
    axes[1].text(v + 0.05, i, f'{v:.2f}', va='center', fontsize=9)

plt.suptitle('Recovery Score Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/07_recovery_score.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Readmission Risk Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
risk_order = ['Low', 'Medium', 'High']
risk_colors = ['#4caf50', '#ff9800', '#f44336']

# Overall distribution
risk_counts = df['readmission_risk'].value_counts().reindex(risk_order)
axes[0].bar(risk_counts.index, risk_counts.values, color=risk_colors)
axes[0].set_title('Readmission Risk Distribution')
axes[0].set_xlabel('Risk Category')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(risk_counts.values):
    pct = v / len(df) * 100
    axes[0].text(i, v + 20, f'{v}\n({pct:.1f}%)', ha='center', fontsize=9)

# Risk by department (stacked)
risk_dept = df.groupby(['department', 'readmission_risk']).size().unstack(fill_value=0)
risk_dept = risk_dept[risk_order]
risk_dept_pct = risk_dept.div(risk_dept.sum(axis=1), axis=0) * 100
risk_dept_pct.plot(kind='bar', stacked=True, ax=axes[1], color=risk_colors, edgecolor='white')
axes[1].set_title('Risk Distribution by Department')
axes[1].set_xlabel('Department')
axes[1].set_ylabel('% of Patients')
axes[1].legend(title='Risk', bbox_to_anchor=(1.01, 1))
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Readmission Risk Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/08_readmission_risk.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. Correlation Heatmap

In [ ]:
num_df = df[['length_of_stay_days', 'treatment_cost', 'recovery_score']]
corr = num_df.corr()

plt.figure(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numerical Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/09_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nCorrelation values:')
print(corr.round(3))

---
## 9. Scatter Plots — Relationships Between Variables

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost vs LOS
axes[0].scatter(df['length_of_stay_days'], df['treatment_cost'],
                alpha=0.3, s=10, color='steelblue')
axes[0].set_title('Treatment Cost vs Length of Stay')
axes[0].set_xlabel('Length of Stay (days)')
axes[0].set_ylabel('Treatment Cost (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

# Cost vs Recovery
axes[1].scatter(df['treatment_cost'], df['recovery_score'],
                alpha=0.3, s=10, color='teal')
axes[1].set_title('Treatment Cost vs Recovery Score')
axes[1].set_xlabel('Treatment Cost (USD)')
axes[1].set_ylabel('Recovery Score')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.suptitle('Variable Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/charts/10_scatter_plots.png', dpi=120, bbox_inches='tight')
plt.show()

print('Note: Near-zero correlations indicate no strong linear relationship between these variables.')